<a href="https://colab.research.google.com/github/antonDinkov/AI_integrationsForDev/blob/main/exerciseOpenAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPARATION

In [2]:
!pip install -q openai

In [3]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 13.2 MB/s eta 0:00:00


In [53]:
from openai import OpenAI
from anthropic import Anthropic
from google.colab import userdata
from pydantic import BaseModel, Field
from pprint import pprint
from pathlib import Path

openai_key = userdata.get('OPEN_AI_API_KEY')
anthropic_key = userdata.get('ANTHROPIC_API_KEY')

openai_client = OpenAI(api_key=openai_key)
anthropic_client = Anthropic(api_key=anthropic_key)

def get_anthropic_message_lines(message):
    return [x.text for x in message.content if x.type == 'text']


def print_anthropic_message(message):
    print(f"Message id: {message.id}")
    print(f"Input: {message.usage.input_tokens}; Output: {message.usage.output_tokens}")
    print(f"Stop reason: {message.stop_reason}")
    pprint(message.content)

    print()
    thinking_content_elements = [x.thinking for x in message.content if x.type == 'thinking']
    if len(thinking_content_elements) > 0:
        print(f"{'-' * 20} [Thinking] {'-' * 20}")
        for line in thinking_content_elements:
            print(line)

    print()
    print(f"{'-' * 20} [Text] {'-' * 20}")
    for line in get_anthropic_message_lines(message):
        print(line)

def print_openai_response(response):
    print(f"Response id: {response.id}")
    print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached); Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
    pprint(response.output)

    print()
    print(f"{'-' * 20} [Text] {'-' * 20}")
    print(response.output_text)

# Upload to Anthropic

In [6]:
path_to_file = Path("/content/test_doc_01.pdf")

with path_to_file.open("rb") as file_content:
  file_upload_response = anthropic_client.beta.files.upload(
      file=(path_to_file.name, file_content, "application/pdf")
  )

In [7]:
pprint(file_upload_response)

FileMetadata(id='file_011CckT1bTLEgLK4CabxXqdz', created_at=datetime.datetime(2026, 7, 6, 7, 58, 40, 798000, tzinfo=datetime.timezone.utc), filename='test_doc_01.pdf', mime_type='application/pdf', size_bytes=52939, type='file', downloadable=False, scope=None)


## Summarize the document

In [14]:
system_prompt = """
You are an expert summarizer of legal and technical documents.

Your task:
- Produce a concise summary of approximately 200 words.
- The summary must preserve the meaning and structure of the source text.

Hard constraints:
- Do NOT invent, assume, or infer any information not explicitly stated in the source document.
- Every statement in the summary must be directly supported by the source text.
- Do NOT add external knowledge, interpretation, or explanations.
- Do NOT introduce new entities, facts, dates, or numbers not present in the input.

Content rules:
- Preserve factual accuracy and neutrality.
- Merge and compress repeated ideas only if they are semantically identical in the source.
- Keep legal/technical meaning intact (do not simplify in a way that changes meaning).
- Remove redundancy, filler, and verbose phrasing.

Output rules:
- Target length: ~200 words (acceptable range: 180–220 words).
- Maintain formal, objective tone.

Priority rule:
- If there is any conflict between conciseness and faithfulness to the source text, faithfulness ALWAYS takes priority.
"""

file_id = file_upload_response.id

summery_response = anthropic_client.beta.messages.create(
    model="claude-haiku-4-5-20251001",
    messages=[
        {"role": "user", "content": [
            {"type": "text", "text": "Summarize the referenced document."},
            {"type": "document", "source": {
                "file_id": file_id, "type": "file"
            }}
        ]}
    ],
    max_tokens=1024,
    system=system_prompt,
    betas=["files-api-2025-04-14"]
)

In [15]:
summery_response

BetaMessage(id='msg_01Cm17yQoQqajcSS9PJjHzNv', container=None, content=[BetaTextBlock(citations=None, text='# Invoice Summary\n\n**Document:** Invoice No. 2026-00147 issued by Try at Software ООД (Shumen, Bulgaria) on 21.02.2026 to ДигиМаркет ЕООД (Plovdiv).\n\n**Issuer Details:** Try at Software ООД, EIK: 207654321, VAT ID: BG207654321, Phone: +359 2 987 6543\n\n**Recipient Details:** ДигиМаркет ЕООД, Boulevard Vitosha 128, Plovdiv 4000, EIK: 305123456, VAT ID: BG305123456\n\n**Services Rendered:**\n1. Web design and corporate website development (4,500.00 BGN)\n2. SEO optimization—initial package (3 months) (1,200.00 BGN)\n3. Premium hosting plan—annual (480.00 BGN)\n4. Graphic design—logo and brand identity (1,800.00 BGN)\n5. Technical support—monthly subscription (3 months at 350.00 BGN each = 1,050.00 BGN)\n6. Wildcard SSL certificate—annual (320.00 BGN)\n7. Payment system integration (Stripe) (750.00 BGN)\n\n**Financial Summary:**\n- Tax Base: 10,100.00 BGN\n- VAT (20%): 2,020.00

In [16]:
print_anthropic_message(summery_response)

Message id: msg_01Cm17yQoQqajcSS9PJjHzNv
Input: 2525; Output: 429
Stop reason: end_turn
[BetaTextBlock(citations=None, text='# Invoice Summary\n\n**Document:** Invoice No. 2026-00147 issued by Try at Software ООД (Shumen, Bulgaria) on 21.02.2026 to ДигиМаркет ЕООД (Plovdiv).\n\n**Issuer Details:** Try at Software ООД, EIK: 207654321, VAT ID: BG207654321, Phone: +359 2 987 6543\n\n**Recipient Details:** ДигиМаркет ЕООД, Boulevard Vitosha 128, Plovdiv 4000, EIK: 305123456, VAT ID: BG305123456\n\n**Services Rendered:**\n1. Web design and corporate website development (4,500.00 BGN)\n2. SEO optimization—initial package (3 months) (1,200.00 BGN)\n3. Premium hosting plan—annual (480.00 BGN)\n4. Graphic design—logo and brand identity (1,800.00 BGN)\n5. Technical support—monthly subscription (3 months at 350.00 BGN each = 1,050.00 BGN)\n6. Wildcard SSL certificate—annual (320.00 BGN)\n7. Payment system integration (Stripe) (750.00 BGN)\n\n**Financial Summary:**\n- Tax Base: 10,100.00 BGN\n- VA

In [20]:
document_summary = "\n".join(get_anthropic_message_lines(summery_response))

In [34]:
document_summary

'# Invoice Summary\n\n**Document:** Invoice No. 2026-00147 issued by Try at Software ООД (Shumen, Bulgaria) on 21.02.2026 to ДигиМаркет ЕООД (Plovdiv).\n\n**Issuer Details:** Try at Software ООД, EIK: 207654321, VAT ID: BG207654321, Phone: +359 2 987 6543\n\n**Recipient Details:** ДигиМаркет ЕООД, Boulevard Vitosha 128, Plovdiv 4000, EIK: 305123456, VAT ID: BG305123456\n\n**Services Rendered:**\n1. Web design and corporate website development (4,500.00 BGN)\n2. SEO optimization—initial package (3 months) (1,200.00 BGN)\n3. Premium hosting plan—annual (480.00 BGN)\n4. Graphic design—logo and brand identity (1,800.00 BGN)\n5. Technical support—monthly subscription (3 months at 350.00 BGN each = 1,050.00 BGN)\n6. Wildcard SSL certificate—annual (320.00 BGN)\n7. Payment system integration (Stripe) (750.00 BGN)\n\n**Financial Summary:**\n- Tax Base: 10,100.00 BGN\n- VAT (20%): 2,020.00 BGN\n- **Total Amount Due: 12,120.00 BGN**\n\n**Payment Details:** ProBank AD, IBAN: BG42PROB9100103254789

In [22]:
delete_file = anthropic_client.beta.files.delete(
    file_id
)

In [23]:
delete_file

DeletedFile(id='file_011CckT1bTLEgLK4CabxXqdz', type='file_deleted')

# Keyword Extraction Using OpenAI API


In [33]:
class KeywordExtractionResult(BaseModel):
  keywords: list[str]

In [35]:
openai_models = openai_client.models.list()

In [37]:
openai_models.data

[Model(id='text-embedding-ada-002', created=1671217299, object='model', owned_by='openai-internal'),
 Model(id='whisper-1', created=1677532384, object='model', owned_by='openai-internal'),
 Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai'),
 Model(id='tts-1', created=1681940951, object='model', owned_by='openai-internal'),
 Model(id='gpt-3.5-turbo-16k', created=1683758102, object='model', owned_by='openai-internal'),
 Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai'),
 Model(id='gpt-4', created=1687882411, object='model', owned_by='openai'),
 Model(id='davinci-002', created=1692634301, object='model', owned_by='system'),
 Model(id='babbage-002', created=1692634615, object='model', owned_by='system'),
 Model(id='gpt-3.5-turbo-instruct', created=1692901427, object='model', owned_by='system'),
 Model(id='gpt-3.5-turbo-instruct-0914', created=1694122472, object='model', owned_by='system'),
 Model(id='gpt-3.5-turbo-1106', created=16

In [54]:
response_openai_keywords = openai_client.responses.parse(
    model="gpt-5.4-nano-2026-03-17",
    input=document_summary,
    text_format=KeywordExtractionResult,
    instructions="You are an expert in keyword extraction. Given a document summary, return a list (maximum 10 items) of the most important keywords. Focus on terms, named entities and domain-specific vocabulary.",
    reasoning={"effort": "medium"}
)

In [55]:
response_openai_keywords

ParsedResponse[TypeVar](id='resp_0e99f8a530bbcded006a4b7f01dae081a3b771352fe3292f91', created_at=1783332609.0, error=None, incomplete_details=None, instructions='You are an expert in keyword extraction. Given a document summary, return a list (maximum 10 items) of the most important keywords. Focus on terms, named entities and domain-specific vocabulary.', metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_0e99f8a530bbcded006a4b7f029d6481a3bd181ae750456620', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"keywords":["Invoice 2026-00147","Try at Software ООД","ДигиМаркет ЕООД","Web design and corporate website development","SEO optimization (initial package, 3 months)","Graphic design (logo and brand identity)","Premium hosting plan (annual)","Wildcard SSL certificate (annual)","Stripe payment system integration","VAT 20% (BGN)"]}', type='output_text', logprobs=[], parsed=KeywordExtractionResult(keywor

In [56]:
print_openai_response(response_openai_keywords)

Response id: resp_0e99f8a530bbcded006a4b7f01dae081a3b771352fe3292f91
Input tokens: 457 (0 cached); Output tokens: 90 (0 reasoning)
[ParsedResponseOutputMessage[TypeVar](id='msg_0e99f8a530bbcded006a4b7f029d6481a3bd181ae750456620', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"keywords":["Invoice 2026-00147","Try at Software ООД","ДигиМаркет ЕООД","Web design and corporate website development","SEO optimization (initial package, 3 months)","Graphic design (logo and brand identity)","Premium hosting plan (annual)","Wildcard SSL certificate (annual)","Stripe payment system integration","VAT 20% (BGN)"]}', type='output_text', logprobs=[], parsed=KeywordExtractionResult(keywords=['Invoice 2026-00147', 'Try at Software ООД', 'ДигиМаркет ЕООД', 'Web design and corporate website development', 'SEO optimization (initial package, 3 months)', 'Graphic design (logo and brand identity)', 'Premium hosting plan (annual)', 'Wildcard SSL certificate (annual)', 'Stripe payment system

In [57]:
pprint(response_openai_keywords)

ParsedResponse[TypeVar](id='resp_0e99f8a530bbcded006a4b7f01dae081a3b771352fe3292f91', created_at=1783332609.0, error=None, incomplete_details=None, instructions='You are an expert in keyword extraction. Given a document summary, return a list (maximum 10 items) of the most important keywords. Focus on terms, named entities and domain-specific vocabulary.', metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ParsedResponseOutputMessage[TypeVar](id='msg_0e99f8a530bbcded006a4b7f029d6481a3bd181ae750456620', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"keywords":["Invoice 2026-00147","Try at Software ООД","ДигиМаркет ЕООД","Web design and corporate website development","SEO optimization (initial package, 3 months)","Graphic design (logo and brand identity)","Premium hosting plan (annual)","Wildcard SSL certificate (annual)","Stripe payment system integration","VAT 20% (BGN)"]}', type='output_text', logprobs=[], parsed=KeywordExtractionResult(keywor

## Categorization

In [58]:
class CategorizationResult(BaseModel):
    category: str
    confidence: float
    short_explanation: str

In [59]:
categorization_response = openai_client.responses.parse(
    model="gpt-5-mini",
    input=document_summary,
    instructions="You are an expert in document categorization. Based on the provided document summary, classify the document into exactly one category. Return the category, a configence score (from 0.0 to 1.0), and a short explanatory statement (1-2 sentences).",
    reasoning={ "effort": "high" },
    text_format=CategorizationResult
)

In [60]:
print_openai_response(categorization_response)

Response id: resp_0aa177703994ec10006a4b7f0f784881a09955207230bfd947
Input tokens: 491 (0 cached); Output tokens: 479 (384 reasoning)
[ResponseReasoningItem(id='rs_0aa177703994ec10006a4b7f0fbea081a0b6c223f807be72c4', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ParsedResponseOutputMessage[TypeVar](id='msg_0aa177703994ec10006a4b7f12d6bc81a0a30b87dcc5f25711', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"category":"Invoice","confidence":0.99,"short_explanation":"This document is an invoice: it includes an invoice number, issuer and recipient details, itemized services with prices, VAT and total amount due, plus payment instructions and a payment deadline."}', type='output_text', logprobs=[], parsed=CategorizationResult(category='Invoice', confidence=0.99, short_explanation='This document is an invoice: it includes an invoice number, issuer and recipient details, itemized services with prices, VAT and total amount due, plus payment 